In [1]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, KFold

## Оптмизиаця признакового пространства

In [2]:
data = pd.read_csv("../data/processed_smoke_detector.csv")
X = data.drop(labels=["Fire Alarm"], axis=1)
y = data["Fire Alarm"]

In [3]:
from sklearn.feature_selection import SelectKBest

In [4]:
skb = SelectKBest(k=2)
X_skb = skb.fit_transform(X, y)
X_skb = pd.DataFrame(X_skb, columns=skb.get_feature_names_out())
X_skb.head()

,TVOC[ppb],Raw Ethanol
0,19.0,19951.0
1,1.0,19975.0
2,10.0,19955.0
3,10.0,19963.0
4,13.0,19958.0


In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_skb, y, test_size=0.2, random_state=42)

## Sklearn MLP

In [7]:
import numpy as np
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from scipy.stats import loguniform
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# 1. RandomizedSearchCV
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50,50), (100,50), (100,100)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'sgd', 'lbfgs'],
    'alpha': loguniform(1e-5, 1e-1),
    'learning_rate_init': loguniform(1e-4, 0.1),
    'batch_size': [16, 32, 64, 128]
}

mlp = MLPClassifier(max_iter=500, random_state=42)
random_search = RandomizedSearchCV(mlp, param_dist, n_iter=20, cv=5, 
                                  scoring='accuracy', n_jobs=-1)
random_search.fit(X_train_clf, y_train_clf)

print("RandomizedSearchCV results:")
print(f"Best params: {random_search.best_params_}")
print(f"Best CV accuracy: {random_search.best_score_:.4f}")
print(f"Test accuracy: {random_search.score(X_test_clf, y_test_clf):.4f}")

# 2. Optuna
def objective(trial):
    params = {
        'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', 
                            [(50,), (100,), (50,50), (100,50), (100,100)]),
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh', 'logistic']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd', 'lbfgs']),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 0.1, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64, 128]),
        'max_iter': 500
    }
    model = MLPClassifier(**params, random_state=42)
    score = cross_val_score(model, X_train_clf, y_train_clf, cv=5, 
                          scoring='accuracy', n_jobs=-1).mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("\nOptuna results:")
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# 3. Hyperopt
space = {
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', 
                    [(50,), (100,), (50,50), (100,50), (100,100)]),
    'activation': hp.choice('activation', ['relu', 'tanh', 'logistic']),
    'solver': hp.choice('solver', ['adam', 'sgd', 'lbfgs']),
    'alpha': hp.loguniform('alpha', np.log(1e-5), np.log(1e-1)),
    'learning_rate_init': hp.loguniform('lr', np.log(1e-4), np.log(0.1)),
    'batch_size': hp.choice('batch_size', [16, 32, 64, 128]),
}

def objective_hyperopt(params):
    model = MLPClassifier(**params, max_iter=500, random_state=42)
    score = cross_val_score(model, X_train_clf, y_train_clf, cv=5, 
                          scoring='accuracy', n_jobs=-1).mean()
    return {'loss': -score, 'status': STATUS_OK}

trials = Trials()
best = fmin(fn=objective_hyperopt, space=space, algo=tpe.suggest, 
           max_evals=20, trials=trials)

print("\nHyperopt results:")
print(f"Best params: {best}")
best_params = {
    'hidden_layer_sizes': [(50,), (100,), (50,50), (100,50), (100,100)][best['hidden_layer_sizes']],
    'activation': ['relu', 'tanh', 'logistic'][best['activation']],
    'solver': ['adam', 'sgd', 'lbfgs'][best['solver']],
    'alpha': best['alpha'],
    'learning_rate_init': best['lr'],
    'batch_size': [16, 32, 64, 128][best['batch_size']]
}
print(f"Decoded params: {best_params}")

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.


## Keras + Tensor Flow FCNN